In [1]:
import sys
import numpy as np

from keras.models import Sequential
from keras.layers import Conv2D
from keras.layers import MaxPooling2D
from keras.layers import Dense
from keras.layers import Flatten
from keras.layers import Dropout
from keras.optimizers import SGD
from keras.losses import CategoricalCrossentropy

sys.path.append('../../../Rain/')
from Rain import Rain
sys.path.pop()

'../../../Rain/'

In [2]:
config = {
    "lib": "tensorflow",
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 50,
    "batch_size": 64,
    "loss": CategoricalCrossentropy(),
    "optimizer": SGD(learning_rate=0.001, momentum=0.9)
}

In [3]:
def get_train_data():
    return np.load('../../../data/CIFAR10/train_data.npy'), np.load('../../../data/CIFAR10/train_labels.npy')

def get_test_data():
    return np.load('../../../data/CIFAR10/test_data.npy'), np.load('../../../data/CIFAR10/test_labels.npy')

def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)
    
    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size:])
            y_train_partitions.append(y_train[i * partition_size:])
        else:
            X_train_partitions.append(X_train[i * partition_size:(i + 1) * partition_size])
            y_train_partitions.append(y_train[i * partition_size:(i + 1) * partition_size])

    return X_train_partitions, y_train_partitions

In [4]:
def create_model():
    model = Sequential()
    model.add(Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same', input_shape=(32, 32, 3)))
    model.add(Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))
    model.add(Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
    model.add(Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))
    model.add(Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
    model.add(Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))
    model.add(Flatten())
    model.add(Dense(128, activation='relu', kernel_initializer='he_uniform'))
    model.add(Dropout(0.2))
    model.add(Dense(10, activation='softmax'))
    return model

In [5]:
X_train, y_train = get_train_data()
X_train, y_train = partition_train_data(X_train, y_train, config['partitions'])

In [9]:
# from keras.models import load_model
# weights = load_model('model.h5').get_weights()
model = create_model()
# model.set_weights(weights)
rain = Rain(config, model, X_train, y_train)
model = rain.train_centralized_sync()

Server listening on port 5000
Connected by ('127.0.0.1', 51199)
Connected by ('127.0.0.1', 51200)
Connected by ('127.0.0.1', 51201)
All workers have connected.
Keras weights file (<HDF5 file "variables.h5" (mode r+)>) saving:
...layers\conv2d
......vars
.........0
.........1
...layers\conv2d_1
......vars
.........0
.........1
...layers\conv2d_2
......vars
.........0
.........1
...layers\conv2d_3
......vars
.........0
.........1
...layers\conv2d_4
......vars
.........0
.........1
...layers\conv2d_5
......vars
.........0
.........1
...layers\dense
......vars
.........0
.........1
...layers\dense_1
......vars
.........0
.........1
...layers\dropout
......vars
...layers\dropout_1
......vars
...layers\dropout_2
......vars
...layers\dropout_3
......vars
...layers\flatten
......vars
...layers\max_pooling2d
......vars
...layers\max_pooling2d_1
......vars
...layers\max_pooling2d_2
......vars
...vars
Keras model archive saving:
File Name                                             Modified      

In [10]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

157/157 [==============================] - 4s 27ms/step - loss: 0.6825 - accuracy: 0.7937

Test accuracy: 79.4%
